# DOH 골프 3D 회전 — NLF 버전 (최신 Colab 대응)

mmpose는 최신 Colab(py3.12·torch2.11)에서 사망 → **NLF**(TorchScript, 토치버전 안 탐)로 교체.

**하는 법:** 위에서부터 회색칸 **▶** 순서대로.
**먼저:** 런타임 → 런타임 유형 변경 → **T4 GPU** → 저장.

> NLF는 비상업 연구용 라이선스 — 지금은 '검증'이라 OK. 서비스화 땐 별도 검토.


### 1칸. 모델 준비 (1~2분)
끝에 **`>>> NLF OK`** 뜨면 성공.


In [ ]:
import torch, torchvision, os
import torchvision.ops                      # NLF 모델이 torchvision::nms 를 써서 등록 필요
print('torch', torch.__version__, '| tv', torchvision.__version__, '| cuda', torch.version.cuda)
URL='https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript'
M='nlf_l_multi_0.3.2.torchscript'
if (not os.path.exists(M)) or os.path.getsize(M) < 10_000_000:
    !wget -q '{URL}' -O '{M}'
sz = os.path.getsize(M)/1e6 if os.path.exists(M) else 0
print('모델 MB:', round(sz,1))
if sz < 10:
    print('❌ 다운로드 실패 — 캡처해서 알려주세요')
else:
    model = torch.jit.load(M).cuda().eval()
    print('>>> NLF OK')


### 2칸. 스윙 영상 올리기


In [ ]:
from google.colab import files
up = files.upload()
VIDEO = list(up.keys())[0]
print('올린 영상:', VIDEO)


### 3칸. 3D 분석 (제일 오래 걸림)
첫 프레임에서 결과 구조를 한 번 찍어봐요(디버그). 이상하면 그 출력만 보내주면 바로 고침.


In [ ]:
import cv2, numpy as np, pickle, torch
cap = cv2.VideoCapture(VIDEO)
J = []; dbg = True
with torch.inference_mode():
    while True:
        ok, fr = cap.read()
        if not ok: break
        rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        t = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).cuda()   # [1,3,H,W] uint8
        pred = model.detect_smpl_batched(t)
        if dbg:
            print('pred keys:', list(pred.keys()) if hasattr(pred,'keys') else type(pred))
            jj = pred['joints3d']
            print('joints3d:', type(jj), 'len', len(jj) if hasattr(jj,'__len__') else '?')
            dbg = False
        per_img = pred['joints3d'][0]        # 첫(유일) 이미지의 검출들
        if per_img is None or len(per_img) == 0:
            continue
        kp = per_img[0]                       # 첫 사람
        kp = kp.detach().cpu().numpy() if torch.is_tensor(kp) else np.asarray(kp)
        J.append(kp)
cap.release()
J = np.array(J)
pickle.dump({'joints': J}, open('joints.pkl','wb'))
print('저장 완료 · frames', J.shape)


### 4칸. 관절 수 확인 (K)
`--check` 출력에서 관절 수(K)를 본다. **K=24 → `smpl`**, K=17 → `h36m`.
(metrics 지표는 현재 smpl에서만 나옴. h36m이면 회전만.)

In [ ]:
BR='claude/ai-video-analysis-engine-wlr06k'
BASE=f'https://raw.githubusercontent.com/tinyalex3628-dotcom/doh-golf-survey/{BR}/pose3d_poc'
!wget -q {BASE}/wham_golf_rotation.py -O rot.py
!wget -q {BASE}/wham_golf_metrics.py  -O wham_golf_metrics.py
!python rot.py joints.pkl --check     # ← 관절 수(K) 확인

In [ ]:
# ── 이 영상 정보 (영상마다 바꾸기) ──
VIEW = 'FO'      # 정면 = 'FO'  /  측면(뒤에서) = 'DTL'
HAND = 'right'   # 오른손잡이 = 'right'  /  왼손잡이 = 'left'

import cv2, os, json
_c = cv2.VideoCapture(VIDEO); FPS = round(_c.get(cv2.CAP_PROP_FPS) or 60, 2); _c.release()
print('fps', FPS, '| view', VIEW, '| hand', HAND)

# 회전그래프 + analyzer2용 rotation.json + doh.vision.v1 계약(회전+지표)
!python rot.py joints.pkl --skeleton smpl --png rot.png \
    --json doh_vision.json \
    --json-v1 doh_vision_v1.json --view {VIEW} --hand {HAND} --fps {FPS}

from IPython.display import Image
if os.path.exists('rot.png'): display(Image('rot.png'))

# 계약 요약 (보고 이상하면 형한테)
d = json.load(open('doh_vision_v1.json'))
print('\n== doh.vision.v1 ==  events:', {e['p']: e['frame'] for e in d['swing_events']})
for f in d['features']:
    print(f"  {f['feature_id']:6s} {f['name'][:36]:36s} {f['value']}")

# 파일명에 view 붙여 저장 → 형한테 이거 보내기
out = f"doh_vision_v1_{VIEW}_{os.path.splitext(VIDEO)[0]}.json"
os.replace('doh_vision_v1.json', out)
from google.colab import files; files.download(out)

### 5칸. 형한테 보낼 것 (영상마다)
- **`doh_vision_v1_...json`** 파일 (제일 중요 — 이거 하나면 됨)
- **그래프 rot.png** 화면
- (있으면) "이 스윙 대충 어깨회전 몇 도" 눈대중

**여러 영상 돌리기:** 위 **2칸(업로드)** 부터 다시 실행 → 영상 올리고 → 3·4칸 → 이 칸.
영상마다 **VIEW 바꾸기**(정면 FO / 측면 DTL). 모델(1칸)은 다시 안 해도 됨.

정면 2~3개 + 측면 2~3개면 충분. 다른 사람 영상이어도 OK.